# 第08課 - 多代理設計模式


## 設置


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 為什麼選擇多智能體系統？

現實世界的任務如行程規劃涉及許多不同種類的專業知識——物流、當地知識、預算等等。單一智能體嘗試處理所有事情很快就會變得難以管理。

多智能體系統透過**專門化**來解決這個問題：每個智能體專注於一個專業領域，產出比一般通才更高品質的結果。它們還提升了**可擴展性**——你可以新增新的智能體（例如，飛行專家、餐廳評論家）而無需重寫現有的工作流程。這些智能體通過結構化的流程管道組合在一起，將上下文從一個傳遞到下一個。


## 創建專門代理人


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## 建立順序工作流程

`WorkflowBuilder` 讓你將代理連接成有向圖。這裡我們建立一個簡單的兩步驟流程：**TravelPlanner** 草擬行程，然後 **TravelConcierge** 審核並加強它。


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## 為工作流程新增更多代理程式

多代理模式其中一個最大的優點是擴展的簡易性。以下我們加入一個 **BudgetReviewer** 代理程式，負責檢查計劃是否符合旅客的預算，標示可能超出預算限制的項目，並建議省錢的替代方案。工作流程現在會依序執行三個代理程式：

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Summary

在這課中你學會了如何：

1. **創建專門的代理** — 每個都有專注的角色（規劃、禮賓、預算審核）。
2. **使用 `WorkflowBuilder` 和 `add_edge` 將代理連接成順序工作流程**。
3. **從多代理流程中串流輸出，追蹤哪個代理在發言**。
4. **擴充工作流程**，透過加入新的代理到流程中而不修改現有的代理。

多代理設計模式保持每個代理的簡單性，同時產生比單一代理更豐富、更全面審核的結果。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：  
本文件係使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 所翻譯。雖然我哋致力確保準確性，但請注意，自動翻譯可能包含錯誤或不準確之處。原始文件嘅母語版本應視為權威來源。對於重要資訊，建議採用專業人工翻譯。對於因使用本翻譯所引起嘅任何誤解或誤譯，我哋概不負責。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
